# Fig. 10: Latent Space PCA

This notebook performs Principal Component Analysis (PCA) on the 32-dimensional latent mean
vectors from the CVAE encoder for both the Holton-Mass model and the Emulator predictions.

It reproduces **Figure 10** from the paper: projection of latent mean vectors onto the first
two principal components, colored by dynamical regime label (AA, BB, AB, BA).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
import pandas as pd
import seaborn as sns
import os
import sys

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

# ============================================================
# PATHS — update these to match your environment
# ============================================================
# Holton-Mass trajectories: shape (N, 2, 75) or (N, 75) depending on your format
DATA_HM_PATH = r"/home/danyul/ssw/stochastic_trajectories.npy"
# Emulator predictions: shape (N, 75) — denormalized
DATA_PRED_PATH = r"/home/danyul/ssw/predictions_best_checkpoint_and_cycle_Resnet_VAE_3mil.npy"
# Trained CVAE model weights
MODEL_PATH = r"/home/danyul/ssw/best_model.pth"

# Add parent directory so we can import model.py
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), ".."))
from model import ConditionalVAE

# ============================================================
# Model setup
# ============================================================
latent_dim = 32
output_dim = 75
condition_dim = 50  # Psi(t) = Re{Psi} + Im{Psi}, 25+25

model = ConditionalVAE(latent_dim, output_dim, condition_dim)
model = model.cuda()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cuda"))
model.eval()
print("Model loaded.")

# ============================================================
# Regime thresholds (from paper Eqs. 2–3, after normalization)
# ============================================================
upper = 53.8 / 2.8935   # u_A threshold (normalized)
lower = 7.41             # u_B threshold (normalized)

# ============================================================
# Transition detection helpers
# ============================================================
def detect_transitions_A_to_B(u_series, upper, lower):
    """Find indices where a genuine A->B transition begins."""
    transitions = []
    i = 1
    while i < len(u_series) - 1:
        if u_series[i - 1] > upper and u_series[i] <= upper:
            j = i + 1
            while j < len(u_series) and u_series[j] <= upper:
                if u_series[j] < lower:
                    transitions.append(i)
                    break
                j += 1
            i = j
        else:
            i += 1
    return np.array(transitions)


def detect_transitions_B_to_A(u_series, upper, lower):
    """Find indices where a genuine B->A transition begins."""
    transitions = []
    i = 1
    while i < len(u_series) - 1:
        if u_series[i - 1] < lower and u_series[i] >= lower:
            j = i + 1
            while j < len(u_series) and u_series[j] >= lower:
                if u_series[j] > upper:
                    transitions.append(i)
                    break
                j += 1
            i = j
        else:
            i += 1
    return np.array(transitions)


def classify_and_sample(zonal_wind, data_array, upper, lower, total_len,
                        hm_format=False):
    """
    Classify each timestep into AA/BB/AB/BA and return balanced samples.

    Parameters
    ----------
    zonal_wind : 1-D array of U(30 km) values
    data_array : full state array to index into for encoding
    upper, lower : regime thresholds
    total_len : number of timesteps to use
    hm_format : if True, data_array has shape (N, 2, 75) and we use [:,0,:]

    Returns
    -------
    X : (4*n_samples, 75) array of states
    labels : list of category strings
    n_samples : number of samples per category
    """
    transition_indices_A = detect_transitions_A_to_B(zonal_wind, upper, lower)
    transition_indices_B = detect_transitions_B_to_A(zonal_wind, upper, lower)
    print(f"  Transitions | A->B: {len(transition_indices_A)}, B->A: {len(transition_indices_B)}")

    transition_indices = np.union1d(transition_indices_A, transition_indices_B)

    non_trans_A = np.where(
        (zonal_wind > upper) & (~np.isin(np.arange(total_len), transition_indices))
    )[0]
    non_trans_B = np.where(
        (zonal_wind < lower) & (~np.isin(np.arange(total_len), transition_indices))
    )[0]
    print(f"  Non-transition | A: {len(non_trans_A)}, B: {len(non_trans_B)}")

    n_samples = min(len(transition_indices_A), len(transition_indices_B),
                    len(non_trans_A), len(non_trans_B))
    print(f"  Balanced samples per class: {n_samples}")

    rng = np.random.default_rng(2025)
    non_trans_A = rng.choice(non_trans_A, size=n_samples, replace=False)
    non_trans_B = rng.choice(non_trans_B, size=n_samples, replace=False)

    if hm_format:
        X = np.vstack([
            data_array[transition_indices_A, 0],
            data_array[transition_indices_B, 0],
            data_array[non_trans_A, 0],
            data_array[non_trans_B, 0],
        ]).astype(np.float32)
    else:
        X = np.vstack([
            data_array[transition_indices_A],
            data_array[transition_indices_B],
            data_array[non_trans_A],
            data_array[non_trans_B],
        ]).astype(np.float32)

    labels = (
        ["AB"] * len(transition_indices_A)
        + ["BA"] * len(transition_indices_B)
        + ["AA"] * len(non_trans_A)
        + ["BB"] * len(non_trans_B)
    )
    return X, labels, n_samples


def encode_and_pca(X, labels, model, n_components=3):
    """Encode states through the CVAE encoder and run PCA on latent means."""
    with torch.no_grad():
        mu, logvar = model.encode(torch.tensor(X, dtype=torch.float32).cuda())
        mu_np = mu.cpu().numpy()

    pca = PCA(n_components=n_components)
    latent = pca.fit_transform(mu_np)

    explained = pca.explained_variance_ratio_
    print(f"  Explained variance: PC1={explained[0]:.4f}, PC2={explained[1]:.4f}, PC3={explained[2]:.4f}")
    print(f"  Total (PC1+2+3): {sum(explained):.4f}")

    df = pd.DataFrame(latent, columns=[f"PC{i+1}" for i in range(n_components)])
    df["Category"] = labels
    return df, pca


# ============================================================
# Color / style settings (matching paper Figure 10)
# ============================================================
colors = {"AB": "green", "BA": "purple", "AA": "blue", "BB": "red"}
category_order = ["BB", "BA", "AB", "AA"]
MAX_POINTS = 750


def interleave_by_category(df_in, order):
    """Interleave points from different categories for better visibility."""
    groups = [df_in[df_in["Category"] == c] for c in order if c in df_in["Category"].unique()]
    indices = [g.index.tolist() for g in groups]
    interleaved = []
    max_len = max((len(lst) for lst in indices), default=0)
    for i in range(max_len):
        for lst in indices:
            if i < len(lst):
                interleaved.append(lst[i])
    return df_in.loc[interleaved]


def plot_pc1_vs_pc2(df, title, save_path=None):
    """Produce the PC1 vs PC2 scatter plot (paper Figure 10 style)."""
    centroids = df.groupby("Category")[["PC1", "PC2"]].mean().reset_index()

    df_plot = df.groupby("Category", group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), MAX_POINTS), random_state=42)
    )
    df_plot = interleave_by_category(df_plot, category_order)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        df_plot["PC1"], df_plot["PC2"],
        s=18, alpha=0.4,
        c=[colors[c] for c in df_plot["Category"]],
        zorder=2,
    )
    ax.scatter(
        centroids["PC1"], centroids["PC2"],
        marker="x", s=150, linewidths=3, color="black",
        label="Centroid", zorder=4,
    )
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

    handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=colors[c],
               markersize=6, label=c)
        for c in category_order if c in df_plot["Category"].values
    ]
    handles.append(
        Line2D([0], [0], marker="x", color="black", markersize=8,
               label="Centroid", linestyle="None")
    )
    ax.legend(handles=handles, fontsize=9)
    ax.grid(True)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


# ============================================================
# 1) Holton-Mass Model
# ============================================================
print("=" * 60)
print("Holton-Mass Model")
print("=" * 60)
data_hm = np.load(DATA_HM_PATH)
total_len_hm = 2_500_000
zonal_wind_hm = data_hm[:total_len_hm, 0, 63]

X_hm, labels_hm, _ = classify_and_sample(
    zonal_wind_hm, data_hm[:total_len_hm], upper, lower, total_len_hm, hm_format=True
)
df_hm, pca_hm = encode_and_pca(X_hm, labels_hm, model)
plot_pc1_vs_pc2(df_hm, "Holton-Mass", save_path="latent_pca_holton_mass.png")

# ============================================================
# 2) Emulator
# ============================================================
print("=" * 60)
print("Emulator")
print("=" * 60)
data_pred = np.load(DATA_PRED_PATH)
total_len_pred = min(2_500_000, len(data_pred))
zonal_wind_pred = data_pred[:total_len_pred, 63]

X_pred, labels_pred, _ = classify_and_sample(
    zonal_wind_pred, data_pred[:total_len_pred], upper, lower, total_len_pred, hm_format=False
)
df_pred, pca_pred = encode_and_pca(X_pred, labels_pred, model)
plot_pc1_vs_pc2(df_pred, "Emulator", save_path="latent_pca_emulator.png")

# ============================================================
# Side-by-side figure (paper Figure 10)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df, ttl in [(axes[0], df_hm, "Holton-Mass"), (axes[1], df_pred, "Emulator")]:
    centroids = df.groupby("Category")[["PC1", "PC2"]].mean().reset_index()
    df_plot = df.groupby("Category", group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), MAX_POINTS), random_state=42)
    )
    df_plot = interleave_by_category(df_plot, category_order)

    ax.scatter(
        df_plot["PC1"], df_plot["PC2"],
        s=18, alpha=0.4,
        c=[colors[c] for c in df_plot["Category"]],
        zorder=2,
    )
    ax.scatter(
        centroids["PC1"], centroids["PC2"],
        marker="x", s=150, linewidths=3, color="black", zorder=4,
    )
    ax.set_title(ttl)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True)

# Shared legend
handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=colors[c],
           markersize=6, label=c)
    for c in category_order
]
handles.append(
    Line2D([0], [0], marker="x", color="black", markersize=8,
           label="Centroid", linestyle="None")
)
axes[1].legend(handles=handles, fontsize=9, loc="upper right")

plt.tight_layout()
plt.savefig("latent_pca_side_by_side.png", dpi=300, bbox_inches="tight")
plt.show()

print("Done.")